# NERSC Job Submission via the AmSC Python Client

This notebook demonstrates how to submit and manage compute jobs on NERSC resources
(Perlmutter) using the AmSC Python Client's facility integration.

**What you'll do:**
1. Create an AmSC client
2. Connect to the NERSC facility and explore available compute resources
3. Submit a "hello world" job to Perlmutter
4. Monitor the job until completion

**Prerequisites:**
- `amsc-client` installed (with `globus-sdk`)
- A valid NERSC account and project allocation
- A Globus account linked to your NERSC identity
- **IRI API allowlist access** — having a NERSC account is not sufficient. Email [NERSC support](https://help.nersc.gov) with your NERSC username and use case to be added to the API access list. Without this, job submission returns HTTP 401.

**Authentication:**
- NERSC facility endpoints (resource listing, incidents) are **public** — no login needed
- Job submission requires a **Globus login** — the client prints an authorization URL; open it, log in, and paste the returned authorization code into the prompt

## Authentication and validation status

- **Credential domain:** direct NERSC facility access uses an independent facility-native Globus credential. Facility-only work does not require `AMSC_TOKEN` and does not reuse an AmSC Keycard.
- **When authentication happens:** when a protected facility call needs a credential and no usable cached credential exists, the built-in NERSC authenticator prints an authorization URL. Open it, log in with your NERSC identity, then paste the returned authorization code into the prompt.
- **Validation scope:** the examples are statically checked against `amsc-client==0.6.1`, but NERSC authentication, protected reads, and submission have not yet been live-validated by the tutorial maintainers.
- **Submission safety:** keep `SUBMIT_JOB=False` unless your NERSC account, allocation, IRI API access, project, queue, and paths are confirmed.
- **Credential safety:** never paste or print a token in a cell, and clear all outputs before sharing the notebook.


In [ ]:
import os
import time
import amsc_client
print(amsc_client.version_info())
from amsc_client import Client, Resource, Job, ApiError

## Step 1: Create the AmSC Client

The client manages authentication and provides access to all AmSC services.
For this tutorial, we only need facility access — no catalog auth is required.

In [ ]:
# Create a client — no central-service credentials are needed for facility-only access.
# The NERSC authenticator (Globus) is resolved automatically on first call.
client = Client()
print("✅ Client created")

## Step 2: Connect to NERSC and Explore Resources

NERSC is a built-in facility — just call `client.facility("nersc")`.
Resource listing is public (no auth required).

In [ ]:
# Connect to NERSC
nersc = client.facility("nersc")

# Get facility info
info = nersc.info()
print(f"Facility: {info.name}")
print(f"Organization: {getattr(info, 'organization_name', 'N/A')}")

In [ ]:
# List all available resources
resources = nersc.resources()

print(f"Available Resources ({len(resources)}):\n")
print(f"{'Name':12s}  {'Type':10s}  {'Status'}")
print(f"{'─'*12}  {'─'*10}  {'─'*12}")
for r in resources:
    print(f"{r.name:12s}  {r.resource_type:10s}  {r.status}")

In [ ]:
# Get the compute resource (Perlmutter)
compute = nersc.resource("compute")

print(f"Resource: {compute.name}")
print(f"ID:       {compute.id}")
print(f"Type:     {compute.resource_type}")
print(f"Status:   {compute.status}")

## Step 3: Check for Active Incidents

Before submitting a job, it's good practice to check for any active
outages or maintenance windows.

In [ ]:
incidents = nersc.incidents()
print(f"Total incidents: {len(incidents)}")

# Show the 5 most recent
if incidents:
    print(f"\nRecent incidents:")
    for inc in incidents[:5]:
        print( f' - {str(inc.last_modified):20} | {inc.name:25} -> {inc.resolution:20} ')

## Step 4: Configure and Submit a Job

Now we'll submit a simple "hello world" job to Perlmutter.

**Authentication:** if no usable cached NERSC credential exists, submission prints an authorization URL. Open it in a private/incognito window, log in with your NERSC identity, then paste the returned authorization code into the prompt. `amsc-client==0.6.1` requests the required identity scopes and forces a fresh identity-provider login whenever an interactive NERSC authorization is needed.

**Before running:** Update `NERSC_ACCOUNT` below to match your NERSC project allocation.

In [ ]:
# ── Submission gate ──────────────────────────────────────────────────────────
# Set to True only when you have a NERSC allocation and IRI API allowlist access.
SUBMIT_JOB = False

# ── NERSC account parameters (required only when SUBMIT_JOB = True) ──────────
NERSC_USERNAME = os.environ.get("NERSC_USERNAME", "")
NERSC_ACCOUNT  = os.environ.get("NERSC_ACCOUNT", "")
NERSC_QUEUE    = os.environ.get("NERSC_QUEUE", "debug")

if SUBMIT_JOB and not NERSC_USERNAME:
    raise EnvironmentError(
        "SUBMIT_JOB is True but NERSC_USERNAME is not set.\n"
        "  export NERSC_USERNAME='your-nersc-username'"
    )
if SUBMIT_JOB and not NERSC_ACCOUNT:
    raise EnvironmentError(
        "SUBMIT_JOB is True but NERSC_ACCOUNT is not set.\n"
        "  export NERSC_ACCOUNT='your-project-allocation'"
    )

OUTPUT_DIR = "/global/homes/" + NERSC_USERNAME[0] + "/" + NERSC_USERNAME if NERSC_USERNAME else ""

import datetime
RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
JOB_NAME = "amsc-nersc-tutorial-" + RUN_ID

print("SUBMIT_JOB    :", SUBMIT_JOB)
print("NERSC_USERNAME:", NERSC_USERNAME or "(not set)")
print("NERSC_ACCOUNT :", NERSC_ACCOUNT or "(not set)")
print("NERSC_QUEUE   :", NERSC_QUEUE)
print("RUN_ID        :", RUN_ID)
print("JOB_NAME      :", JOB_NAME)
print("OUTPUT_DIR    :", OUTPUT_DIR)


In [ ]:
if SUBMIT_JOB:
    # Submit the job
    # If needed, this call starts the authorization URL/code login flow.
    
    job = compute.submit(
        executable="/bin/echo",
        arguments=["Hello from AmSC Python Client!", RUN_ID],
        directory=OUTPUT_DIR,
        stdout_path=f"{OUTPUT_DIR}/{JOB_NAME}.stdout",
        stderr_path=f"{OUTPUT_DIR}/{JOB_NAME}.stderr",
        name=JOB_NAME,
        queue=NERSC_QUEUE,
        account=NERSC_ACCOUNT,
        duration=300,        # Wall time in seconds (5 minutes)
        nodes=1,             # Number of nodes
    )
    
    print(f"\n✅ Job submitted!")
    print(f"   Job ID:    {job.id}")
    print(f"   State:     {job.state}")
    print(f"   {job!r}")


## Step 5: Monitor the Job

The `Job` object is "self-aware" — it knows which facility and resource it belongs to,
and can refresh its own status from the API.

We can poll manually or use `job.wait()` to block until completion.

In [ ]:
if SUBMIT_JOB and 'job' in dir():
    # Manual polling — check status every 5 seconds
    
    POLL_TIMEOUT = 60  # seconds
    POLL_INTERVAL = 5  # seconds
    
    print(f"Polling job {job.id} (up to {POLL_TIMEOUT}s)...\n")
    start_time = time.time()
    
    while time.time() - start_time < POLL_TIMEOUT:
        elapsed = int(time.time() - start_time)
        try:
            current_status = job.status  # calls the API
            state = job.state
            print(f"  [{elapsed:3d}s] State: {state}")
    
            if job.is_terminal:
                exit_code = job.exit_code
                message = job.message
                print(f"\n✅ Job finished!")
                print(f"   Final state: {state}")
                if exit_code is not None:
                    print(f"   Exit code:   {exit_code}")
                if message:
                    print(f"   Message:     {message}")
                break
        except Exception as e:
            # Completed Slurm jobs may disappear from the queue;
            # the API returns 400 "not found" in that case
            if "not found" in str(e).lower() or "400" in str(e):
                print(f"  [{elapsed:3d}s] Job no longer in scheduler queue (likely completed)")
                print(f"\n✅ Job completed (exited scheduler)")
                break
            else:
                print(f"  [{elapsed:3d}s] Status check error: {e}")
    
        time.sleep(POLL_INTERVAL)
    else:
        print(f"\n⏰ Timed out after {POLL_TIMEOUT}s (job may still be running)")
        print(f"   Last known state: {job.state}")


## Step 6: Read Job Output via the Filesystem API

Now that the job has completed, we can read its output directly
using the **filesystem API** — no need to SSH into the machine.

We use **homes** (NERSC's home filesystem resource) to access the output files,
since job output was written to the user's home directory.

In [ ]:
if SUBMIT_JOB and 'job' in dir():
    # Access the filesystem through the homes resource
    homes = nersc.resource("homes")

    stdout_path = f"{OUTPUT_DIR}/{JOB_NAME}.stdout"
    stderr_path = f"{OUTPUT_DIR}/{JOB_NAME}.stderr"

    # Read the job's stdout
    # Task.wait() is a no-op in 0.6.1 — result is synchronously available
    try:
        task = homes.fs.head(stdout_path, lines=50)
        print(f"── {stdout_path} ──")
        print(task.result)
    except ApiError as e:
        print(f"🚧 head: Not available yet — {e}")

    # Read the job's stderr (should be empty for a successful job)
    try:
        task = homes.fs.head(stderr_path, lines=50)
        print(f"\n── {stderr_path} ──")
        print(task.result if task.result else "(empty)")
    except ApiError as e:
        print(f"🚧 head: Not available yet — {e}")

## Alternative: Using `job.wait()`

Instead of manual polling, you can use the built-in `wait()` method
which blocks until the job reaches a terminal state:

In [ ]:
if SUBMIT_JOB:
    # Submit and wait for completion
    
    job2 = compute.submit(
        executable="/bin/hostname",
        directory=OUTPUT_DIR,
        stdout_path=f"{OUTPUT_DIR}/amsc-wait-demo-{RUN_ID}.stdout",
        stderr_path=f"{OUTPUT_DIR}/amsc-wait-demo-{RUN_ID}.stderr",
        name=f"amsc-wait-demo-{RUN_ID}",
        queue=NERSC_QUEUE,
        account=NERSC_ACCOUNT,
        duration=300,
        nodes=1,
    )
    print(f"✅ Job submitted: {job2.id}")
    print(f"   Waiting for completion...")
    
    try:
        job2.wait(timeout=360, poll_interval=10)
        print(f"✅ Job completed: state={job2.state}, exit_code={job2.exit_code}")
    except TimeoutError:
        print(f"⏳ Job still running after timeout (state: {job2.state})")


## Summary

This tutorial showed how to:

| Step | API Call | Auth Required? |
|------|---------|----------------|
| Connect to NERSC | `client.facility("nersc")` | No |
| List resources | `nersc.resources()` | No |
| Get resource details | `nersc.resource("compute")` | No |
| Check incidents | `nersc.incidents()` | No |
| Submit a job | `compute.submit(...)` | Yes (Globus → NERSC) |
| Check job status | `job.status` | Yes |
| Wait for completion | `job.wait(timeout=120)` | Yes |
| Read job output | `homes.fs.head(path)` | Yes |
| Cancel a job | `job.cancel()` | Yes |

**Key concepts:**
- **Resource-scoped compute**: Jobs are submitted via the resource object (`compute.submit(...)`) rather than a separate compute client
- **Self-aware jobs**: The `Job` object tracks its own facility and resource, and can refresh status, wait, or cancel itself
- **Filesystem via homes**: Use `homes.fs.head()` to read job output files without SSH — homes sees the same home directories as compute nodes
- **Lazy authentication**: Globus login is only triggered when you first call an authenticated endpoint (e.g., `submit()`)
- **Slurm behavior**: Completed Slurm jobs may disappear from the scheduler queue; the API returns a 400 error when this happens

## Troubleshooting

### 401 on Job Submission (Not on IRI API Allowlist)

If you get an `HTTP 401` error when submitting a job (after successfully logging in), your account may not be on the NERSC IRI API access list. Having a NERSC account and allocation is not enough — access to the IRI API must be requested separately. Email [NERSC support](https://help.nersc.gov) with your NERSC username and use case.

### `session_info.authentications: {}` After Re-Authentication

This means Globus issued an IRI token, but NERSC did not receive evidence of a fresh identity-provider authentication. It is distinct from missing IRI allowlist access.

1. Stop the notebook kernel.
2. Move `~/.amsc/credentials.json` aside rather than opening or printing it. This shared cache can contain credentials for more than one service.
3. Restart the kernel and run the notebook again.
4. Open the printed authorization URL in a private/incognito window and explicitly log in with your NERSC identity.

`amsc-client==0.6.1` requests the NERSC IRI and identity scopes and uses `prompt=login` whenever an interactive authorization flow is needed. If this error persists, confirm that you explicitly selected and authenticated with your NERSC identity.